# N4: Optimizing Single MLP (Model 1 from N3)

Optimize the baseline MLP from N3 with different configurations:
1. **Model 4.1**: Original architecture with class weighting
2. **Model 4.2**: Deeper network (512-256-128-64)
3. **Model 4.3**: Thinner network (128-64)
4. **Model 4.4**: With L2 regularization
5. **Model 4.5**: Different learning rate (0.0001)

## 1. Import Required Libraries

In [1]:
import numpy as np
import pandas as pd
import warnings
import torch
import gc
from tqdm import tqdm
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score, precision_score, recall_score
from sklearn.neural_network import MLPClassifier
from transformers import AutoTokenizer, AutoModel

# Setup device for PyTorch (PhoBERT extraction)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("✅ All imports successful")
print(f"   PyTorch Device: {device}")
print(f"   Testing: Class Weighted + Regularized MLPs")

✅ All imports successful
   PyTorch Device: cuda
   Testing: Class Weighted + Regularized MLPs


## 2. Load Data from CSV

In [2]:
train_path = '../../../data/splited/train_set.csv'
val_path = '../../../data/splited/val_set.csv'

df_train = pd.read_csv(train_path)
df_val = pd.read_csv(val_path)

y_train = df_train['label'].values
y_val = df_val['label'].values

print(f"✅ Data loaded: Train {df_train.shape} | Val {df_val.shape}")
print(f"   Labels: Train {(y_train==0).sum()} fake / {(y_train==1).sum()} real")
print(f"           Val {(y_val==0).sum()} fake / {(y_val==1).sum()} real")

# Calculate class weights for handling imbalance
from sklearn.utils.class_weight import compute_class_weight
class_weights = compute_class_weight('balanced', classes=np.unique(y_train), y=y_train)
class_weight_dict = dict(enumerate(class_weights))
print(f"\n📊 Class Weights (balanced): {class_weight_dict}")

✅ Data loaded: Train (3788, 28) | Val (474, 28)
   Labels: Train 3143 fake / 645 real
           Val 393 fake / 81 real

📊 Class Weights (balanced): {0: 0.60260897231944, 1: 2.936434108527132}


## 3. Generate TF-IDF Embeddings

In [3]:
texts_train_strict = df_train['text_strict'].fillna('').tolist()
texts_val_strict = df_val['text_strict'].fillna('').tolist()

tfidf = TfidfVectorizer(ngram_range=(1, 2), max_df=0.95, min_df=2, sublinear_tf=True)
X_train_tfidf_full = tfidf.fit_transform(texts_train_strict)
X_val_tfidf_full = tfidf.transform(texts_val_strict)

svd = TruncatedSVD(n_components=900, random_state=42)
X_train_tfidf = svd.fit_transform(X_train_tfidf_full)
X_val_tfidf = svd.transform(X_val_tfidf_full)

print(f"✅ TF-IDF embeddings: Train {X_train_tfidf.shape} | Val {X_val_tfidf.shape}")

✅ TF-IDF embeddings: Train (3788, 900) | Val (474, 900)


## 4. Extract PhoBERT Embeddings

In [4]:
texts_train_loose = df_train['text_loose'].fillna('').tolist()
texts_val_loose = df_val['text_loose'].fillna('').tolist()

def extract_phobert_embeddings(texts, batch_size=16):
    """Extract PhoBERT embeddings from text"""
    tokenizer = AutoTokenizer.from_pretrained('vinai/phobert-base-v2')
    model = AutoModel.from_pretrained('vinai/phobert-base-v2').to(device).eval()
    embeddings = []
    
    with torch.no_grad():
        for i in tqdm(range(0, len(texts), batch_size), desc="PhoBERT", leave=False):
            batch = texts[i:i+batch_size]
            inputs = tokenizer(batch, return_tensors='pt', padding=True, truncation=True, max_length=256)
            inputs = {k: v.to(device) for k, v in inputs.items()}
            outputs = model(**inputs, output_hidden_states=True)
            cls_tokens = outputs.last_hidden_state[:, 0, :].cpu().numpy()
            embeddings.extend(cls_tokens)
            del outputs, inputs
            torch.cuda.empty_cache()
    
    model.to('cpu')
    del model, tokenizer
    gc.collect()
    return np.array(embeddings)

X_train_phobert = extract_phobert_embeddings(texts_train_loose, batch_size=16)
X_val_phobert = extract_phobert_embeddings(texts_val_loose, batch_size=16)

print(f"✅ PhoBERT embeddings: Train {X_train_phobert.shape} | Val {X_val_phobert.shape}")

'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 0dfbf49e-928f-442a-9756-6ce46be3db07)')' thrown while requesting HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 2b673a96-29ed-44c7-8b4b-c1eaea6673cd)')' thrown while requesting HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeout=10)"), '(Request ID: 45a8933d-3e18-4863-ae78-e5e5800a20f8)')' thrown while requesting HEAD https://huggingface.co/vinai/phobert-base-v2/resolve/main/tokenizer_config.json
Retrying in 2s [Retry 2/5].
'(ReadTimeoutError("HTTPSConnectionPool(host='huggingface.co', port=443): Read timed out. (read timeo

✅ PhoBERT embeddings: Train (3788, 768) | Val (474, 768)


## 5. Combine Embeddings

In [5]:
# Combine embeddings (TF-IDF + PhoBERT)
X_train_combined = np.hstack([X_train_tfidf, X_train_phobert])
X_val_combined = np.hstack([X_val_tfidf, X_val_phobert])

# Normalize
scaler = StandardScaler()
X_train_combined_scaled = scaler.fit_transform(X_train_combined)
X_val_combined_scaled = scaler.transform(X_val_combined)

print(f"✅ Combined embeddings: {X_train_combined_scaled.shape[1]} dimensions")
print(f"   TF-IDF: 900 + PhoBERT: 768 = Total: {900 + 768}")

✅ Combined embeddings: 1668 dimensions
   TF-IDF: 900 + PhoBERT: 768 = Total: 1668


## 6. Model 4.1: Original (256-128-64) + Class Weighting

In [6]:
print("\n" + "="*80)
print("MODEL 4.1: ORIGINAL ARCHITECTURE + CLASS WEIGHTING")
print("="*80)

print("\nConfiguration:")
print("  Architecture: 256 → 128 → 64")
print("  Activation: ReLU")
print("  Learning rate: 0.001")
print("  Class weights: Balanced")

model_4_1 = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    solver='adam',
    learning_rate='constant',
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    batch_size=32,
    random_state=42,
    verbose=False,
    alpha=0.0  # No L2 regularization
)

print("\n📊 Training Model 4.1...")
model_4_1.fit(X_train_combined_scaled, y_train)

print(f"✅ Training complete!")
print(f"   Converged: {model_4_1.n_iter_} iterations")
print(f"   Loss: {model_4_1.loss_:.4f}")

# Evaluate
y_val_pred_4_1 = model_4_1.predict(X_val_combined_scaled)
y_val_proba_4_1 = model_4_1.predict_proba(X_val_combined_scaled)[:, 1]

results_4_1 = {
    'F1': f1_score(y_val, y_val_pred_4_1, average='weighted'),
    'AUC': roc_auc_score(y_val, y_val_proba_4_1),
    'Accuracy': accuracy_score(y_val, y_val_pred_4_1),
    'Precision': precision_score(y_val, y_val_pred_4_1, average='weighted'),
    'Recall': recall_score(y_val, y_val_pred_4_1, average='weighted')
}

print(f"\n✅ Model 4.1 Results:")
print(f"   F1: {results_4_1['F1']:.4f} | AUC: {results_4_1['AUC']:.4f}")
print(f"   Accuracy: {results_4_1['Accuracy']:.4f} | Precision: {results_4_1['Precision']:.4f} | Recall: {results_4_1['Recall']:.4f}")


MODEL 4.1: ORIGINAL ARCHITECTURE + CLASS WEIGHTING

Configuration:
  Architecture: 256 → 128 → 64
  Activation: ReLU
  Learning rate: 0.001
  Class weights: Balanced

📊 Training Model 4.1...
✅ Training complete!
   Converged: 13 iterations
   Loss: 0.0017

✅ Model 4.1 Results:
   F1: 0.9124 | AUC: 0.9573
   Accuracy: 0.9135 | Precision: 0.9116 | Recall: 0.9135


## 7. Model 4.2: Deeper Network (512-256-128-64)

In [7]:
print("\n" + "="*80)
print("MODEL 4.2: DEEPER NETWORK (512-256-128-64)")
print("="*80)

print("\nConfiguration:")
print("  Architecture: 512 → 256 → 128 → 64")
print("  Activation: ReLU")
print("  Learning rate: 0.001")
print("  L2 Regularization: 0.01")

model_4_2 = MLPClassifier(
    hidden_layer_sizes=(512, 256, 128, 64),
    activation='relu',
    solver='adam',
    learning_rate='constant',
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    batch_size=32,
    random_state=42,
    verbose=False,
    alpha=0.01  # L2 regularization
)

print("\n📊 Training Model 4.2...")
model_4_2.fit(X_train_combined_scaled, y_train)

print(f"✅ Training complete!")
print(f"   Converged: {model_4_2.n_iter_} iterations")
print(f"   Loss: {model_4_2.loss_:.4f}")

# Evaluate
y_val_pred_4_2 = model_4_2.predict(X_val_combined_scaled)
y_val_proba_4_2 = model_4_2.predict_proba(X_val_combined_scaled)[:, 1]

results_4_2 = {
    'F1': f1_score(y_val, y_val_pred_4_2, average='weighted'),
    'AUC': roc_auc_score(y_val, y_val_proba_4_2),
    'Accuracy': accuracy_score(y_val, y_val_pred_4_2),
    'Precision': precision_score(y_val, y_val_pred_4_2, average='weighted'),
    'Recall': recall_score(y_val, y_val_pred_4_2, average='weighted')
}

print(f"\n✅ Model 4.2 Results:")
print(f"   F1: {results_4_2['F1']:.4f} | AUC: {results_4_2['AUC']:.4f}")
print(f"   Accuracy: {results_4_2['Accuracy']:.4f} | Precision: {results_4_2['Precision']:.4f} | Recall: {results_4_2['Recall']:.4f}")


MODEL 4.2: DEEPER NETWORK (512-256-128-64)

Configuration:
  Architecture: 512 → 256 → 128 → 64
  Activation: ReLU
  Learning rate: 0.001
  L2 Regularization: 0.01

📊 Training Model 4.2...
✅ Training complete!
   Converged: 18 iterations
   Loss: 0.1285

✅ Model 4.2 Results:
   F1: 0.9057 | AUC: 0.9353
   Accuracy: 0.9072 | Precision: 0.9048 | Recall: 0.9072


## 8. Model 4.3: Thinner Network (128-64)

In [8]:
print("\n" + "="*80)
print("MODEL 4.3: THINNER NETWORK (128-64)")
print("="*80)

print("\nConfiguration:")
print("  Architecture: 128 → 64")
print("  Activation: ReLU")
print("  Learning rate: 0.001")
print("  Purpose: Test if simpler architecture is sufficient")

model_4_3 = MLPClassifier(
    hidden_layer_sizes=(128, 64),
    activation='relu',
    solver='adam',
    learning_rate='constant',
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    batch_size=32,
    random_state=42,
    verbose=False,
    alpha=0.0
)

print("\n📊 Training Model 4.3...")
model_4_3.fit(X_train_combined_scaled, y_train)

print(f"✅ Training complete!")
print(f"   Converged: {model_4_3.n_iter_} iterations")
print(f"   Loss: {model_4_3.loss_:.4f}")

# Evaluate
y_val_pred_4_3 = model_4_3.predict(X_val_combined_scaled)
y_val_proba_4_3 = model_4_3.predict_proba(X_val_combined_scaled)[:, 1]

results_4_3 = {
    'F1': f1_score(y_val, y_val_pred_4_3, average='weighted'),
    'AUC': roc_auc_score(y_val, y_val_proba_4_3),
    'Accuracy': accuracy_score(y_val, y_val_pred_4_3),
    'Precision': precision_score(y_val, y_val_pred_4_3, average='weighted'),
    'Recall': recall_score(y_val, y_val_pred_4_3, average='weighted')
}

print(f"\n✅ Model 4.3 Results:")
print(f"   F1: {results_4_3['F1']:.4f} | AUC: {results_4_3['AUC']:.4f}")
print(f"   Accuracy: {results_4_3['Accuracy']:.4f} | Precision: {results_4_3['Precision']:.4f} | Recall: {results_4_3['Recall']:.4f}")


MODEL 4.3: THINNER NETWORK (128-64)

Configuration:
  Architecture: 128 → 64
  Activation: ReLU
  Learning rate: 0.001
  Purpose: Test if simpler architecture is sufficient

📊 Training Model 4.3...
✅ Training complete!
   Converged: 14 iterations
   Loss: 0.0023

✅ Model 4.3 Results:
   F1: 0.9194 | AUC: 0.9467
   Accuracy: 0.9198 | Precision: 0.9191 | Recall: 0.9198


## 9. Model 4.4: Original with Strong L2 Regularization

In [9]:
print("\n" + "="*80)
print("MODEL 4.4: STRONG L2 REGULARIZATION (alpha=0.1)")
print("="*80)

print("\nConfiguration:")
print("  Architecture: 256 → 128 → 64")
print("  Activation: ReLU")
print("  Learning rate: 0.001")
print("  L2 Regularization: 0.1 (HIGH)")
print("  Purpose: Test aggressive regularization to prevent overfitting")

model_4_4 = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    solver='adam',
    learning_rate='constant',
    learning_rate_init=0.001,
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    batch_size=32,
    random_state=42,
    verbose=False,
    alpha=0.1  # Strong L2 regularization
)

print("\n📊 Training Model 4.4...")
model_4_4.fit(X_train_combined_scaled, y_train)

print(f"✅ Training complete!")
print(f"   Converged: {model_4_4.n_iter_} iterations")
print(f"   Loss: {model_4_4.loss_:.4f}")

# Evaluate
y_val_pred_4_4 = model_4_4.predict(X_val_combined_scaled)
y_val_proba_4_4 = model_4_4.predict_proba(X_val_combined_scaled)[:, 1]

results_4_4 = {
    'F1': f1_score(y_val, y_val_pred_4_4, average='weighted'),
    'AUC': roc_auc_score(y_val, y_val_proba_4_4),
    'Accuracy': accuracy_score(y_val, y_val_pred_4_4),
    'Precision': precision_score(y_val, y_val_pred_4_4, average='weighted'),
    'Recall': recall_score(y_val, y_val_pred_4_4, average='weighted')
}

print(f"\n✅ Model 4.4 Results:")
print(f"   F1: {results_4_4['F1']:.4f} | AUC: {results_4_4['AUC']:.4f}")
print(f"   Accuracy: {results_4_4['Accuracy']:.4f} | Precision: {results_4_4['Precision']:.4f} | Recall: {results_4_4['Recall']:.4f}")


MODEL 4.4: STRONG L2 REGULARIZATION (alpha=0.1)

Configuration:
  Architecture: 256 → 128 → 64
  Activation: ReLU
  Learning rate: 0.001
  L2 Regularization: 0.1 (HIGH)
  Purpose: Test aggressive regularization to prevent overfitting

📊 Training Model 4.4...
✅ Training complete!
   Converged: 13 iterations
   Loss: 0.1512

✅ Model 4.4 Results:
   F1: 0.9138 | AUC: 0.9594
   Accuracy: 0.9156 | Precision: 0.9129 | Recall: 0.9156


## 10. Model 4.5: Lower Learning Rate (0.0001)

In [10]:
print("\n" + "="*80)
print("MODEL 4.5: LOWER LEARNING RATE (0.0001)")
print("="*80)

print("\nConfiguration:")
print("  Architecture: 256 → 128 → 64")
print("  Activation: ReLU")
print("  Learning rate: 0.0001 (10x LOWER)")
print("  L2 Regularization: 0.01")
print("  Purpose: Test slower learning for better convergence")

model_4_5 = MLPClassifier(
    hidden_layer_sizes=(256, 128, 64),
    activation='relu',
    solver='adam',
    learning_rate='constant',
    learning_rate_init=0.0001,  # Much lower learning rate
    max_iter=300,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    batch_size=32,
    random_state=42,
    verbose=False,
    alpha=0.1
)

print("\n📊 Training Model 4.5...")
model_4_5.fit(X_train_combined_scaled, y_train)

print(f"✅ Training complete!")
print(f"   Converged: {model_4_5.n_iter_} iterations")
print(f"   Loss: {model_4_5.loss_:.4f}")

# Evaluate
y_val_pred_4_5 = model_4_5.predict(X_val_combined_scaled)
y_val_proba_4_5 = model_4_5.predict_proba(X_val_combined_scaled)[:, 1]

results_4_5 = {
    'F1': f1_score(y_val, y_val_pred_4_5, average='weighted'),
    'AUC': roc_auc_score(y_val, y_val_proba_4_5),
    'Accuracy': accuracy_score(y_val, y_val_pred_4_5),
    'Precision': precision_score(y_val, y_val_pred_4_5, average='weighted'),
    'Recall': recall_score(y_val, y_val_pred_4_5, average='weighted')
}

print(f"\n✅ Model 4.5 Results:")
print(f"   F1: {results_4_5['F1']:.4f} | AUC: {results_4_5['AUC']:.4f}")
print(f"   Accuracy: {results_4_5['Accuracy']:.4f} | Precision: {results_4_5['Precision']:.4f} | Recall: {results_4_5['Recall']:.4f}")


MODEL 4.5: LOWER LEARNING RATE (0.0001)

Configuration:
  Architecture: 256 → 128 → 64
  Activation: ReLU
  Learning rate: 0.0001 (10x LOWER)
  L2 Regularization: 0.01
  Purpose: Test slower learning for better convergence

📊 Training Model 4.5...
✅ Training complete!
   Converged: 26 iterations
   Loss: 0.2696

✅ Model 4.5 Results:
   F1: 0.9105 | AUC: 0.9548
   Accuracy: 0.9114 | Precision: 0.9098 | Recall: 0.9114


## 11. Final Comparison

In [11]:
print("\n" + "="*80)
print("FINAL COMPARISON: ALL 5 MODELS")
print("="*80)

comparison_results = {
    'Model': [
        'Model 4.1 (256-128-64)',
        'Model 4.2 (512-256-128-64 + L2_0.01)',
        'Model 4.3 (128-64)',
        'Model 4.4 (256-128-64 + L2_0.1)',
        'Model 4.5 (256-128-64 + LR_0.0001)'
    ],
    'Architecture': [
        '3 layers (256-128-64)',
        '4 layers (512-256-128-64)',
        '2 layers (128-64)',
        '3 layers (256-128-64)',
        '3 layers (256-128-64)'
    ],
    'Key Feature': [
        'Baseline',
        'Deeper + Regularization',
        'Simpler',
        'Strong Regularization',
        'Lower Learning Rate'
    ],
    'F1': [
        f"{results_4_1['F1']:.4f}",
        f"{results_4_2['F1']:.4f}",
        f"{results_4_3['F1']:.4f}",
        f"{results_4_4['F1']:.4f}",
        f"{results_4_5['F1']:.4f}"
    ],
    'AUC': [
        f"{results_4_1['AUC']:.4f}",
        f"{results_4_2['AUC']:.4f}",
        f"{results_4_3['AUC']:.4f}",
        f"{results_4_4['AUC']:.4f}",
        f"{results_4_5['AUC']:.4f}"
    ],
    'Accuracy': [
        f"{results_4_1['Accuracy']:.4f}",
        f"{results_4_2['Accuracy']:.4f}",
        f"{results_4_3['Accuracy']:.4f}",
        f"{results_4_4['Accuracy']:.4f}",
        f"{results_4_5['Accuracy']:.4f}"
    ],
    'Precision': [
        f"{results_4_1['Precision']:.4f}",
        f"{results_4_2['Precision']:.4f}",
        f"{results_4_3['Precision']:.4f}",
        f"{results_4_4['Precision']:.4f}",
        f"{results_4_5['Precision']:.4f}"
    ],
    'Recall': [
        f"{results_4_1['Recall']:.4f}",
        f"{results_4_2['Recall']:.4f}",
        f"{results_4_3['Recall']:.4f}",
        f"{results_4_4['Recall']:.4f}",
        f"{results_4_5['Recall']:.4f}"
    ]
}

comparison_df = pd.DataFrame(comparison_results)
print("\n" + comparison_df.to_string(index=False))

# Find best model
models_list = [results_4_1, results_4_2, results_4_3, results_4_4, results_4_5]
auc_scores = [m['AUC'] for m in models_list]
best_idx = np.argmax(auc_scores)

print(f"\n🏆 Best Model by AUC: {comparison_results['Model'][best_idx]}")
print(f"   AUC: {auc_scores[best_idx]:.4f} | F1: {models_list[best_idx]['F1']:.4f}")

print(f"\n✅ Analysis Complete!")


FINAL COMPARISON: ALL 5 MODELS

                               Model              Architecture             Key Feature     F1    AUC Accuracy Precision Recall
              Model 4.1 (256-128-64)     3 layers (256-128-64)                Baseline 0.9124 0.9573   0.9135    0.9116 0.9135
Model 4.2 (512-256-128-64 + L2_0.01) 4 layers (512-256-128-64) Deeper + Regularization 0.9057 0.9353   0.9072    0.9048 0.9072
                  Model 4.3 (128-64)         2 layers (128-64)                 Simpler 0.9194 0.9467   0.9198    0.9191 0.9198
     Model 4.4 (256-128-64 + L2_0.1)     3 layers (256-128-64)   Strong Regularization 0.9138 0.9594   0.9156    0.9129 0.9156
  Model 4.5 (256-128-64 + LR_0.0001)     3 layers (256-128-64)     Lower Learning Rate 0.9105 0.9548   0.9114    0.9098 0.9114

🏆 Best Model by AUC: Model 4.4 (256-128-64 + L2_0.1)
   AUC: 0.9594 | F1: 0.9138

✅ Analysis Complete!


## 12. Save Best Model to Disk

In [12]:
import joblib
import os
from datetime import datetime

print("\n" + "="*80)
print("SAVING BEST MODEL")
print("="*80)

# Determine best model by AUC
model_candidates = {
    'Model_4_1': (model_4_1, results_4_1),
    'Model_4_2': (model_4_2, results_4_2),
    'Model_4_3': (model_4_3, results_4_3),
    'Model_4_4': (model_4_4, results_4_4),
    'Model_4_5': (model_4_5, results_4_5)
}

best_model_name = max(model_candidates.keys(), 
                      key=lambda x: model_candidates[x][1]['AUC'])
best_model, best_results = model_candidates[best_model_name]

print(f"\n🏆 Best Model Selected: {best_model_name}")
print(f"   AUC: {best_results['AUC']:.4f}")
print(f"   F1:  {best_results['F1']:.4f}")
print(f"   Accuracy: {best_results['Accuracy']:.4f}")

# Create directory if not exists
save_dir = os.path.abspath('../../../model/machine_learned')
os.makedirs(save_dir, exist_ok=True)

# Generate filename with timestamp
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
model_filename = os.path.join(save_dir, f'{best_model_name}_{timestamp}.pkl')
scaler_filename = os.path.join(save_dir, f'scaler_{timestamp}.pkl')
metadata_filename = os.path.join(save_dir, f'{best_model_name}_{timestamp}_metadata.txt')

# Save model
joblib.dump(best_model, model_filename)
print(f"\n✅ Model saved: {model_filename}")

# Save scaler
joblib.dump(scaler, scaler_filename)
print(f"✅ Scaler saved: {scaler_filename}")

# Save metadata
metadata = f"""
Best Model Metadata
====================
Model Name: {best_model_name}
Timestamp: {timestamp}
Model File: {model_filename}
Scaler File: {scaler_filename}

PERFORMANCE METRICS:
--------------------
F1 Score:    {best_results['F1']:.4f}
AUC-ROC:     {best_results['AUC']:.4f}
Accuracy:    {best_results['Accuracy']:.4f}
Precision:   {best_results['Precision']:.4f}
Recall:      {best_results['Recall']:.4f}

MODEL ARCHITECTURE:
-------------------
"""

if best_model_name == 'Model_4_1':
    metadata += "Architecture: 256 → 128 → 64\nRegularization (L2 alpha): 0.0\n"
elif best_model_name == 'Model_4_2':
    metadata += "Architecture: 512 → 256 → 128 → 64\nRegularization (L2 alpha): 0.01\n"
elif best_model_name == 'Model_4_3':
    metadata += "Architecture: 128 → 64\nRegularization (L2 alpha): 0.0\n"
elif best_model_name == 'Model_4_4':
    metadata += "Architecture: 256 → 128 → 64\nRegularization (L2 alpha): 0.1\n"
elif best_model_name == 'Model_4_5':
    metadata += "Architecture: 256 → 128 → 64\nRegularization (L2 alpha): 0.1\nLearning Rate: 0.0001\n"

metadata += f"""
Activation: ReLU
Solver: Adam
Learning Rate: 0.001 (or 0.0001 for Model 4.5)
Max Iterations: 300
Early Stopping: True
Batch Size: 32

INPUT FEATURES:
---------------
TF-IDF: 900 dimensions (SVD-reduced)
PhoBERT: 768 dimensions ([CLS] token from Layer 12)
Total: 1668 dimensions

USAGE:
------
import joblib
model = joblib.load('{model_filename}')
scaler = joblib.load('{scaler_filename}')

# Prepare data
X_scaled = scaler.transform(X_input)

# Make predictions
predictions = model.predict(X_scaled)
probabilities = model.predict_proba(X_scaled)[:, 1]
"""

with open(metadata_filename, 'w', encoding='utf-8') as f:
    f.write(metadata)
print(f"✅ Metadata saved: {metadata_filename}")

print(f"\n" + "="*80)
print("✅ Model Saving Complete!")
print("="*80)
print(f"\nSaved to directory: {save_dir}")
print(f"\nFiles:")
print(f"  1. Model:    {os.path.basename(model_filename)}")
print(f"  2. Scaler:   {os.path.basename(scaler_filename)}")
print(f"  3. Metadata: {os.path.basename(metadata_filename)}")


SAVING BEST MODEL

🏆 Best Model Selected: Model_4_4
   AUC: 0.9594
   F1:  0.9138
   Accuracy: 0.9156

✅ Model saved: d:\Vietnamese-Fake-News-Detection\model\machine_learned\Model_4_4_20260406_192428.pkl
✅ Scaler saved: d:\Vietnamese-Fake-News-Detection\model\machine_learned\scaler_20260406_192428.pkl
✅ Metadata saved: d:\Vietnamese-Fake-News-Detection\model\machine_learned\Model_4_4_20260406_192428_metadata.txt

✅ Model Saving Complete!

Saved to directory: d:\Vietnamese-Fake-News-Detection\model\machine_learned

Files:
  1. Model:    Model_4_4_20260406_192428.pkl
  2. Scaler:   scaler_20260406_192428.pkl
  3. Metadata: Model_4_4_20260406_192428_metadata.txt
